## Import Libraries

In [1]:

# Stage 1: Imports and configuration

import os
import pandas as pd

from dotenv import load_dotenv

# HTML webpage loading
from langchain_community.document_loaders import WebBaseLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Sentence Transformers embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Chroma vector database
from langchain_chroma import Chroma

# Load environment variables from .env
load_dotenv()

print("Imports completed successfully!")


C:\Users\kavya\AppData\Local\Temp\ipykernel_18396\1785620860.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Imports completed successfully!


## Loading HTML Pages

In [2]:

# Stage 2: Load official HTML webpages

from langchain_community.document_loaders import WebBaseLoader

sources = [
    {
        "title": "Floods: How to Protect Your Health",
        "url": "https://www.who.int/news-room/questions-and-answers/item/how-do-i-protect-my-health-in-a-flood/",
        "organization": "WHO",
        "region": "Global",
        "topic": "Floods"
    },
    {
        "title": "Public Health Advice Before, During and After Floods",
        "url": "https://www.who.int/europe/emergencies/situations/climate-crisis-extreme-weather/public-health-advice-on-protecting-your-health-before-during-and-after-floods",
        "organization": "WHO Regional Office for Europe",
        "region": "European Region",
        "topic": "Floods"
    },
    {
        "title": "Heat Waves",
        "url": "https://www.who.int/india/heat-waves",
        "organization": "WHO India",
        "region": "India",
        "topic": "Heatwaves"
    },
    {
        "title": "Heat and Health",
        "url": "https://www.who.int/en/news-room/fact-sheets/detail/climate-change-heat-and-health",
        "organization": "WHO",
        "region": "Global",
        "topic": "Heat and Health"
    },
    {
        "title": "Leptospirosis",
        "url": "https://ncdc.mohfw.gov.in/includes/About/CentresAndDivision/leptospirosis.php",
        "organization": "National Centre for Disease Control, India",
        "region": "India",
        "topic": "Infectious Diseases"
    }
]

documents = []

for source_info in sources:
    try:
        loader = WebBaseLoader(web_paths=[source_info["url"]])
        loaded_docs = loader.load()

        for doc in loaded_docs:
            doc.metadata.update(source_info)
            documents.append(doc)

        print(f"Loaded: {source_info['title']}")

    except Exception as e:
        print(f"Failed: {source_info['title']} — {e}")

print("\nTotal documents loaded:", len(documents))


Loaded: Floods: How to Protect Your Health
Loaded: Public Health Advice Before, During and After Floods
Loaded: Heat Waves
Loaded: Heat and Health
Loaded: Leptospirosis

Total documents loaded: 5


In [3]:

# Stage 2: Verify extracted webpage content

if documents:
    for i, doc in enumerate(documents, start=1):
        print(f"\nDocument {i}")
        print("Title:", doc.metadata["title"])
        print("Organization:", doc.metadata["organization"])
        print("Source:", doc.metadata["url"])
        print("Characters:", len(doc.page_content))
        print("Preview:", doc.page_content[:400])
        print("-" * 70)
else:
    print("No webpages were loaded. Check the URLs and internet connection.")



Document 1
Title: Floods: How to Protect Your Health
Organization: WHO
Source: https://www.who.int/news-room/questions-and-answers/item/how-do-i-protect-my-health-in-a-flood/
Characters: 6151
Preview:       
	Floods: How to protect your health
                     
   Skip to main content       


 







Global


Regions







WHO Regional websites







Africa





Americas





South-East Asia





Europe





Eastern Mediterranean





Western Pacific









   













When autocomplete results are available use up and down arrows to review and enter to select.
















----------------------------------------------------------------------

Document 2
Title: Public Health Advice Before, During and After Floods
Organization: WHO Regional Office for Europe
Source: https://www.who.int/europe/emergencies/situations/climate-crisis-extreme-weather/public-health-advice-on-protecting-your-health-before-during-and-after-floods
Characters: 6627
Preview:       
	Public heal

## Cleaning and Splitting 

In [4]:

# Stage 3.1: Clean webpage text

import re

cleaned_documents = []

for doc in documents:
    text = doc.page_content

    # Remove excessive whitespace and blank lines
    text = re.sub(r"\s+", " ", text).strip()

    # Create a new document with the cleaned text and original metadata
    cleaned_doc = doc.model_copy(
        update={"page_content": text}
    )

    cleaned_documents.append(cleaned_doc)

print("Documents cleaned:", len(cleaned_documents))

for doc in cleaned_documents:
    print(
        doc.metadata["title"],
        "->",
        len(doc.page_content),
        "characters"
    )


Documents cleaned: 5
Floods: How to Protect Your Health -> 5002 characters
Public Health Advice Before, During and After Floods -> 5264 characters
Heat Waves -> 4964 characters
Heat and Health -> 14230 characters
Leptospirosis -> 7537 characters


In [5]:

# Stage 3.2: Split documents into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(cleaned_documents)

print("Original documents:", len(cleaned_documents))
print("Total chunks created:", len(chunks))


Original documents: 5
Total chunks created: 61


In [6]:

# Stage 3.3: Inspect sample chunks

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}")
    print("Source:", chunk.metadata["title"])
    print("Topic:", chunk.metadata["topic"])
    print("Length:", len(chunk.page_content))
    print("Content:", chunk.page_content[:500])
    print("-" * 70)



Chunk 1
Source: Floods: How to Protect Your Health
Topic: Floods
Length: 262
Content: Floods: How to protect your health Skip to main content Global Regions WHO Regional websites Africa Americas South-East Asia Europe Eastern Mediterranean Western Pacific When autocomplete results are available use up and down arrows to review and enter to select
----------------------------------------------------------------------

Chunk 2
Source: Floods: How to Protect Your Health
Topic: Floods
Length: 789
Content: . Select language Select language English العربية 中文 Français Русский Español Português Home Health Topics All topicsABCDEFGHIJKLMNOPQRSTUVWXYZ Resources Fact sheets Facts in pictures Multimedia Podcasts Publications Questions and answers Tools and toolkits Popular Dengue Endometriosis Excessive heat Herpes Mental disorders Mpox Countries All countriesABCDEFGHIJKLMNOPQRSTUVWXYZ Regions Africa Americas Europe Eastern Mediterranean South-East Asia Western Pacific WHO in countries Data by c

## Embedding

In [7]:

# Stage 4.1: Initialize Sentence Transformer embeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence Transformer embedding model initialized!")


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17259.13it/s]


Sentence Transformer embedding model initialized!


In [8]:

# Stage 4.2: Create a persistent Chroma vector database

os.makedirs("vectorstore", exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="public_health_guidelines",
    persist_directory="./vectorstore/chroma_db"
)

print("Chroma vector database created successfully!")
print("Total chunks stored:", vectorstore._collection.count())


Chroma vector database created successfully!
Total chunks stored: 122


In [9]:

# Stage 4.3: Test semantic search

question = "How can people protect themselves from heatwaves?"

results = vectorstore.similarity_search(
    question,
    k=3
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result.metadata["title"])
    print("Organization:", result.metadata["organization"])
    print("Content:", result.page_content[:500])
    print("-" * 70)



Result 1
Source: Heat and Health
Organization: WHO
Content: . Heatwaves can also be associated with hazardous air pollution events. The scale and nature of the health impacts of heat depend on the timing, intensity and duration of a heat event, and the level of acclimatization and adaptability of the local population, infrastructure and institutions to the prevailing climate.Infographic: Scale and nature of the health impacts of heatWhat actions should the public take?Stay out of the heatAvoid going outside and doing strenuous activity during the hottest
----------------------------------------------------------------------

Result 2
Source: Heat and Health
Organization: WHO
Content: . Heatwaves can also be associated with hazardous air pollution events. The scale and nature of the health impacts of heat depend on the timing, intensity and duration of a heat event, and the level of acclimatization and adaptability of the local population, infrastructure and institutions to the prevail

## LLM 

In [10]:

# Stage 5.1: Initialize the Gemini language model

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.1
)

print("Gemini language model initialized!")


Gemini language model initialized!


In [11]:

# Stage 5.2: Create the Basic RAG prompt

basic_rag_prompt = ChatPromptTemplate.from_template("""
You are a Public Health Guidelines Research Assistant.

Answer the user's question using the provided context.

Instructions:
1. Use the context as the primary source of information.
2. Do not invent facts or unsupported medical advice.
3. If the context does not contain enough information, say so clearly.
4. Distinguish India-specific guidance from global or European guidance.
5. Give a clear, concise answer.
6. Do not claim that the retrieved context is a substitute for
   professional medical advice.

Context:
{context}

User question:
{question}

Answer:
""")

basic_rag_chain = basic_rag_prompt | llm | StrOutputParser()

print("Basic RAG prompt and chain created!")


Basic RAG prompt and chain created!


In [12]:

# Stage 5.3: Basic RAG implementation

def basic_rag(question, k=3):
    # Step 1: Retrieve relevant chunks
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=k
    )

    # Step 2: Prepare context
    context = "\n\n".join(
        f"Source: {doc.metadata.get('title', 'Unknown')}\n"
        f"Organization: {doc.metadata.get('organization', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )

    # Step 3: Generate an answer
    answer = basic_rag_chain.invoke({
        "context": context,
        "question": question
    })

    # Step 4: Collect source information
    sources_used = [
        {
            "title": doc.metadata.get("title", "Unknown"),
            "organization": doc.metadata.get("organization", "Unknown"),
            "url": doc.metadata.get("url", "")
        }
        for doc in retrieved_docs
    ]

    return {
        "question": question,
        "answer": answer,
        "sources": sources_used
    }

print("Basic RAG function created!")


Basic RAG function created!


In [13]:

# Stage 5.4: Test Basic RAG

question = "How can people protect themselves during a heatwave?"

response = basic_rag(question)

print("QUESTION:")
print(response["question"])

print("\nGENERATED ANSWER:")
print(response["answer"])

print("\nSOURCES:")
for source in response["sources"]:
    print("-", source["title"])
    print("  Organization:", source["organization"])
    print("  URL:", source["url"])


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Advanced Rag

In [ ]:

# Stage 6.1: Query expansion

query_expansion_prompt = ChatPromptTemplate.from_template("""
You are helping retrieve public health guidance.

Generate 3 alternative search queries for the user's question.
Keep the original meaning. Use different relevant wording.
Return only the 3 queries, one per line, without numbering.

Question: {question}
""")

query_expansion_chain = (
    query_expansion_prompt
    | llm
    | StrOutputParser()
)

def expand_query(question):
    response = query_expansion_chain.invoke({
        "question": question
    })

    alternative_queries = [
        line.strip().lstrip("0123456789.-) ")
        for line in response.splitlines()
        if line.strip()
    ]

    return [question] + alternative_queries[:3]

# Test query expansion
test_queries = expand_query(
    "How can people protect themselves during a heatwave?"
)

for query in test_queries:
    print("-", query)


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


- How can people protect themselves during a heatwave?
- Extreme heat safety tips and precautions
- Ways to stay cool and safe during high temperatures
- Public health advice for heatwave protection


In [ ]:

# Stage 6.2: Multi-query retrieval

def retrieve_candidates(question, k=3):
    queries = expand_query(question)
    candidate_docs = []
    seen = set()

    for query in queries:
        docs = vectorstore.similarity_search(
            query,
            k=k
        )

        for doc in docs:
            key = (
                doc.metadata.get("url", ""),
                doc.page_content
            )

            if key not in seen:
                seen.add(key)
                candidate_docs.append(doc)

    return candidate_docs

# Test retrieval
candidates = retrieve_candidates(
    "How can people protect themselves during a heatwave?"
)

print("Unique candidate passages:", len(candidates))

for i, doc in enumerate(candidates[:5], start=1):
    print(f"\nCandidate {i}")
    print("Source:", doc.metadata.get("title"))
    print("Content:", doc.page_content[:300])


Unique candidate passages: 6

Candidate 1
Source: Heat and Health
Content: . Heatwaves can also be associated with hazardous air pollution events. The scale and nature of the health impacts of heat depend on the timing, intensity and duration of a heat event, and the level of acclimatization and adaptability of the local population, infrastructure and institutions to the p

Candidate 2
Source: Heat and Health
Content: . Health professionals should adjust their guidance, planning and interventions to account for increasing heat exposures, as well as to manage acute increases in admissions associated with heatwaves. Practical, feasible and often low-cost interventions at the individual, community, organizational, g

Candidate 3
Source: Heat and Health
Content: . WHO has issued guidance for public health institutions to identify and manage extreme heat risks. Action on climate change combined with comprehensive preparedness and risk management can save lives now and in the future. Overvie

In [ ]:

# Stage 6.3: Rerank candidate passages

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank_documents(question, candidate_docs, top_n=3):
    if not candidate_docs:
        return []

    pairs = [
        (question, doc.page_content)
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(candidate_docs, scores),
        key=lambda item: item[1],
        reverse=True
    )

    return [
        (doc, float(score))
        for doc, score in ranked_docs[:top_n]
    ]

# Test reranking
ranked_results = rerank_documents(
    "How can people protect themselves during a heatwave?",
    candidates,
    top_n=3
)

for i, (doc, score) in enumerate(ranked_results, start=1):
    print(f"\nRank {i} | Score: {score:.4f}")
    print("Source:", doc.metadata.get("title"))
    print("Content:", doc.page_content[:300])


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2597.57it/s]



Rank 1 | Score: 0.7809
Source: Heat and Health
Content: . Heatwaves can also be associated with hazardous air pollution events. The scale and nature of the health impacts of heat depend on the timing, intensity and duration of a heat event, and the level of acclimatization and adaptability of the local population, infrastructure and institutions to the p

Rank 2 | Score: 0.3352
Source: Heat and Health
Content: . Health professionals should adjust their guidance, planning and interventions to account for increasing heat exposures, as well as to manage acute increases in admissions associated with heatwaves. Practical, feasible and often low-cost interventions at the individual, community, organizational, g

Rank 3 | Score: 0.1160
Source: Heat and Health
Content: . WHO has issued guidance for public health institutions to identify and manage extreme heat risks. Action on climate change combined with comprehensive preparedness and risk management can save lives now and in the future. Ove

In [ ]:

# Stage 6.4: Advanced RAG implementation

def advanced_rag(question, k=3, top_n=3):
    # Step 1: Expand the query and retrieve candidates
    candidates = retrieve_candidates(question, k=k)

    # Step 2: Rerank candidates
    ranked_results = rerank_documents(
        question,
        candidates,
        top_n=top_n
    )

    # Step 3: Prepare the best-ranked context
    context = "\n\n".join(
        f"Source: {doc.metadata.get('title', 'Unknown')}\n"
        f"Organization: {doc.metadata.get('organization', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc, score in ranked_results
    )

    if not context:
        return {
            "question": question,
            "answer": "No relevant passages were retrieved. Please try rephrasing your question.",
            "sources": []
        }

    # Step 4: Generate an answer using the same grounded prompt
    answer = basic_rag_chain.invoke({
        "context": context,
        "question": question
    })

    # Step 5: Return unique sources
    sources_used = []
    seen_urls = set()

    for doc, score in ranked_results:
        url = doc.metadata.get("url", "")

        if url not in seen_urls:
            seen_urls.add(url)
            sources_used.append({
                "title": doc.metadata.get("title", "Unknown"),
                "organization": doc.metadata.get("organization", "Unknown"),
                "url": url,
                "reranking_score": round(score, 4)
            })

    return {
        "question": question,
        "answer": answer,
        "sources": sources_used
    }

print("Advanced RAG function created!")


Advanced RAG function created!


In [ ]:

# Stage 6.5: Test Advanced RAG

question = "How can people protect themselves during a heatwave?"

response = advanced_rag(question)

print("QUESTION:")
print(response["question"])

print("\nGENERATED ANSWER:")
print(response["answer"])

print("\nRERANKED SOURCES:")
for source in response["sources"]:
    print("-", source["title"])
    print("  Organization:", source["organization"])
    print("  Reranking score:", source["reranking_score"])
    print("  URL:", source["url"])


QUESTION:
How can people protect themselves during a heatwave?

GENERATED ANSWER:
Based on global guidance from the World Health Organization (WHO) provided in the context, individuals can protect themselves during a heatwave by taking the following actions:

* **Stay out of the heat:** Avoid going outside and refrain from strenuous physical activity during the hottest time of the day.
* **Stay in the shade:** Keep in mind that perceived temperatures in direct sunlight can be 10–15°C higher.
* **Spend time in cool spaces:** Spend 2–3 hours during the day in a cool place.
* **Practice water safety:** Be mindful of the risk of drowning and never swim alone.

*(Note: The provided source material contains global guidance from the WHO; it does not include specific guidance differentiated for India or Europe. This information is for educational purposes and is not a substitute for professional medical advice.)*

RERANKED SOURCES:
- Heat and Health
  Organization: WHO
  Reranking score: 0.780

## Evaluation

In [ ]:

# Stage 7.1: Create the evaluation dataset

evaluation_data = [
    {
        "question": "How can people protect themselves during a flood?",
        "expected_topic": "Floods",
        "reference_answer": "Follow official flood warnings, avoid floodwater, and follow public-health and safety guidance."
    },
    {
        "question": "What precautions should people take before a flood?",
        "expected_topic": "Floods",
        "reference_answer": "Prepare for flooding, follow official warnings, and plan how to stay safe."
    },
    {
        "question": "What health risks can arise after a flood?",
        "expected_topic": "Floods",
        "reference_answer": "Floods can expose people to contaminated water, injuries, and other health hazards."
    },
    
]

evaluation_df = pd.DataFrame(evaluation_data)

print("Evaluation questions:", len(evaluation_df))
display(evaluation_df)


Evaluation questions: 3


,question,expected_topic,reference_answer
0,How can people protect themselves during a flood?,Floods,"Follow official flood warnings, avoid floodwat..."
1,What precautions should people take before a f...,Floods,"Prepare for flooding, follow official warnings..."
2,What health risks can arise after a flood?,Floods,Floods can expose people to contaminated water...


In [ ]:

# Stage 7.2: Evaluate retrieval quality

def evaluate_basic_retrieval(question, expected_topic, k=3):
    docs = vectorstore.similarity_search(question, k=k)

    relevant_ranks = [
        rank
        for rank, doc in enumerate(docs, start=1)
        if doc.metadata.get("topic") == expected_topic
    ]

    return {
        "hit": int(bool(relevant_ranks)),
        "reciprocal_rank": (
            1 / min(relevant_ranks)
            if relevant_ranks else 0
        )
    }


def evaluate_advanced_retrieval(question, expected_topic, k=3, top_n=3):
    candidates = retrieve_candidates(question, k=k)
    ranked_docs = rerank_documents(
        question,
        candidates,
        top_n=top_n
    )

    relevant_ranks = [
        rank
        for rank, (doc, score) in enumerate(ranked_docs, start=1)
        if doc.metadata.get("topic") == expected_topic
    ]

    return {
        "hit": int(bool(relevant_ranks)),
        "reciprocal_rank": (
            1 / min(relevant_ranks)
            if relevant_ranks else 0
        )
    }


print("Retrieval evaluation functions created!")


Retrieval evaluation functions created!


In [ ]:

# Stage 7.3: Evaluate both retrieval approaches

evaluation_rows = []

for index, row in evaluation_df.iterrows():
    question = row["question"]
    expected_topic = row["expected_topic"]

    basic_result = evaluate_basic_retrieval(
        question,
        expected_topic
    )

    advanced_result = evaluate_advanced_retrieval(
        question,
        expected_topic
    )

    evaluation_rows.append({
        "question": question,
        "expected_topic": expected_topic,
        "basic_hit": basic_result["hit"],
        "basic_rr": basic_result["reciprocal_rank"],
        "advanced_hit": advanced_result["hit"],
        "advanced_rr": advanced_result["reciprocal_rank"]
    })

    print(f"Evaluated {index + 1}/{len(evaluation_df)}")

retrieval_results_df = pd.DataFrame(evaluation_rows)

print("\nRetrieval evaluation completed!")
display(retrieval_results_df)


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Evaluated 1/3


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Evaluated 2/3


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Evaluated 3/3

Retrieval evaluation completed!


,question,expected_topic,basic_hit,basic_rr,advanced_hit,advanced_rr
0,How can people protect themselves during a flood?,Floods,1,1.0,1,1.0
1,What precautions should people take before a f...,Floods,1,1.0,1,1.0
2,What health risks can arise after a flood?,Floods,1,1.0,1,1.0


In [ ]:

# Stage 7.4: Calculate retrieval metrics

metrics_df = pd.DataFrame([
    {
        "Version": "Basic RAG",
        "Hit Rate@3": retrieval_results_df["basic_hit"].mean(),
        "MRR": retrieval_results_df["basic_rr"].mean()
    },
    {
        "Version": "Advanced RAG",
        "Hit Rate@3": retrieval_results_df["advanced_hit"].mean(),
        "MRR": retrieval_results_df["advanced_rr"].mean()
    }
])

metrics_df["Hit Rate@3 (%)"] = (
    metrics_df["Hit Rate@3"] * 100
).round(2)

metrics_df["MRR"] = metrics_df["MRR"].round(4)

print("Retrieval performance comparison:")
display(
    metrics_df[["Version", "Hit Rate@3 (%)", "MRR"]]
)


Retrieval performance comparison:


,Version,Hit Rate@3 (%),MRR
0,Basic RAG,100.0,1.0
1,Advanced RAG,100.0,1.0


In [ ]:

# Stage 8.1: Enable LangSmith tracing

load_dotenv(override=True)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Public-Health-RAG"

if not os.getenv("LANGSMITH_API_KEY"):
    print("LangSmith API key is missing. Check your .env file.")
else:
    print("LangSmith tracing is configured.")
    print("Project:", os.environ["LANGSMITH_PROJECT"])


LangSmith tracing is configured.
Project: Public-Health-RAG


## Evaluate Answer Quality

In [ ]:

# Stage 9.1: Generate answers for evaluation

answer_evaluation_rows = []

for index, row in evaluation_df.iterrows():
    question = row["question"]

    basic_response = basic_rag(question)
    advanced_response = advanced_rag(question)

    answer_evaluation_rows.append({
        "question": question,
        "expected_topic": row["expected_topic"],
        "reference_answer": row["reference_answer"],
        "basic_answer": basic_response["answer"],
        "basic_sources": basic_response["sources"],
        "advanced_answer": advanced_response["answer"],
        "advanced_sources": advanced_response["sources"]
    })

    print(f"Generated answers for {index + 1}/{len(evaluation_df)} questions")

answer_results_df = pd.DataFrame(answer_evaluation_rows)

print("Answer generation evaluation completed!")
display(
    answer_results_df[
        ["question", "basic_answer", "advanced_answer"]
    ].head(5)
)


c:\Users\kavya\OneDrive\Desktop\advanced rag\health\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:

# Stage 9.2: Check whether expected topics appear in sources

def expected_topic_found(sources, expected_topic):
    return int(any(
        source.get("topic", expected_topic) == expected_topic
        for source in sources
    ))


In [ ]:

# Stage 9.2 (corrected): Source-topic accuracy

url_to_topic = {
    source["url"]: source["topic"]
    for source in sources
}

def expected_topic_found(source_list, expected_topic):
    return int(any(
        url_to_topic.get(source.get("url")) == expected_topic
        for source in source_list
    ))

answer_results_df["basic_source_topic_hit"] = answer_results_df.apply(
    lambda row: expected_topic_found(
        row["basic_sources"], row["expected_topic"]
    ),
    axis=1
)

answer_results_df["advanced_source_topic_hit"] = answer_results_df.apply(
    lambda row: expected_topic_found(
        row["advanced_sources"], row["expected_topic"]
    ),
    axis=1
)

print("Basic source-topic accuracy:",
      round(answer_results_df["basic_source_topic_hit"].mean() * 100, 2), "%")

print("Advanced source-topic accuracy:",
      round(answer_results_df["advanced_source_topic_hit"].mean() * 100, 2), "%")


In [ ]:

# Stage 9.3: Token-overlap evaluation

import re

def token_f1(reference, prediction):
    reference_tokens = re.findall(
        r"\b\w+\b", reference.lower()
    )
    prediction_tokens = re.findall(
        r"\b\w+\b", prediction.lower()
    )

    if not reference_tokens or not prediction_tokens:
        return 0.0

    reference_counts = {}
    for token in reference_tokens:
        reference_counts[token] = reference_counts.get(token, 0) + 1

    prediction_counts = {}
    for token in prediction_tokens:
        prediction_counts[token] = prediction_counts.get(token, 0) + 1

    overlap = sum(
        min(count, prediction_counts.get(token, 0))
        for token, count in reference_counts.items()
    )

    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


answer_results_df["basic_answer_f1"] = answer_results_df.apply(
    lambda row: token_f1(
        row["reference_answer"], row["basic_answer"]
    ),
    axis=1
)

answer_results_df["advanced_answer_f1"] = answer_results_df.apply(
    lambda row: token_f1(
        row["reference_answer"], row["advanced_answer"]
    ),
    axis=1
)

answer_metrics_df = pd.DataFrame([
    {
        "Version": "Basic RAG",
        "Source-topic accuracy (%)":
            answer_results_df["basic_source_topic_hit"].mean() * 100,
        "Mean answer token F1":
            answer_results_df["basic_answer_f1"].mean()
    },
    {
        "Version": "Advanced RAG",
        "Source-topic accuracy (%)":
            answer_results_df["advanced_source_topic_hit"].mean() * 100,
        "Mean answer token F1":
            answer_results_df["advanced_answer_f1"].mean()
    }
])

answer_metrics_df["Source-topic accuracy (%)"] = (
    answer_metrics_df["Source-topic accuracy (%)"].round(2)
)
answer_metrics_df["Mean answer token F1"] = (
    answer_metrics_df["Mean answer token F1"].round(4)
)

display(answer_metrics_df)
